In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
#from sklearn.manifold import TSNE
from scipy import integrate as int
from scipy import stats
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn import datasets
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
import statsmodels.stats.weightstats as ws
from sklearn.cluster import KMeans
import umap
from lmfit import minimize, Parameters
import matplotlib 
from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler


import matplotlib.pyplot as pl
from sklearn.cluster import AgglomerativeClustering

# define a bunch of functions 

In [ ]:
def wfall(shap_values, max_display=10, show=True):
    """ Plots an explantion of a single prediction as a waterfall plot.
    The SHAP value of a feature represents the impact of the evidence provided by that feature on the model's
    output. The waterfall plot is designed to visually display how the SHAP values (evidence) of each feature
    move the model output from our prior expectation under the background data distribution, to the final model
    prediction given the evidence of all the features. Features are sorted by the magnitude of their SHAP values
    with the smallest magnitude features grouped together at the bottom of the plot when the number of features
    in the models exceeds the max_display parameter.
    
    Parameters
    ----------
    shap_values : Explanation
        A one-dimensional Explanation object that contains the feature values and SHAP values to plot.
    max_display : str
        The maximum number of features to plot.
    show : bool
        Whether matplotlib.pyplot.show() is called before returning. Setting this to False allows the plot
        to be customized further after it has been created.
    """
    dark_o= mpl.colors.to_rgb('dimgray')
    dim_g= mpl.colors.to_rgb('darkorange')

    base_values = shap_values.base_values
    
    features = shap_values.data
    feature_names = shap_values.feature_names
    lower_bounds = getattr(shap_values, "lower_bounds", None)
    upper_bounds = getattr(shap_values, "upper_bounds", None)
    values = shap_values.values

    # make sure we only have a single output to explain
    if (type(base_values) == np.ndarray and len(base_values) > 0) or type(base_values) == list:
        raise Exception("waterfall_plot requires a scalar base_values of the model output as the first " \
                        "parameter, but you have passed an array as the first parameter! " \
                        "Try shap.waterfall_plot(explainer.base_values[0], values[0], X[0]) or " \
                        "for multi-output models try " \
                        "shap.waterfall_plot(explainer.base_values[0], values[0][0], X[0]).")

    # make sure we only have a single explanation to plot
    if len(values.shape) == 2:
        raise Exception("The waterfall_plot can currently only plot a single explanation but a matrix of explanations was passed!")
    
    # unwrap pandas series
    if safe_isinstance(features, "pandas.core.series.Series"):
        if feature_names is None:
            feature_names = list(features.index)
        features = features.values

    # fallback feature names
    if feature_names is None:
        feature_names = np.array([labels['FEATURE'] % str(i) for i in range(len(values))])
    
    # init variables we use for tracking the plot locations
    num_features = min(max_display, len(values))
    row_height = 0.5
    rng = range(num_features - 1, -1, -1)
    order = np.argsort(-np.abs(values))
    pos_lefts = []
    pos_inds = []
    pos_widths = []
    pos_low = []
    pos_high = []
    neg_lefts = []
    neg_inds = []
    neg_widths = []
    neg_low = []
    neg_high = []
    loc = base_values + values.sum()
    yticklabels = ["" for i in range(num_features + 1)]
    
    # size the plot based on how many features we are plotting
    pl.gcf().set_size_inches(8, num_features * row_height + 1.5)

    # see how many individual (vs. grouped at the end) features we are plotting
    if num_features == len(values):
        num_individual = num_features
    else:
        num_individual = num_features - 1

    # compute the locations of the individual features and plot the dashed connecting lines
    for i in range(num_individual):
        sval = values[order[i]]
        loc -= sval
        if sval >= 0:
            pos_inds.append(rng[i])
            pos_widths.append(sval)
            if lower_bounds is not None:
                pos_low.append(lower_bounds[order[i]])
                pos_high.append(upper_bounds[order[i]])
            pos_lefts.append(loc)
        else:
            neg_inds.append(rng[i])
            neg_widths.append(sval)
            if lower_bounds is not None:
                neg_low.append(lower_bounds[order[i]])
                neg_high.append(upper_bounds[order[i]])
            neg_lefts.append(loc)
        if num_individual != num_features or i + 4 < num_individual:
            pl.plot([loc, loc], [rng[i] -1 - 0.4, rng[i] + 0.4], color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
        if features is None:
            yticklabels[rng[i]] = feature_names[order[i]]
        else:
            yticklabels[rng[i]] = format_value(features[order[i]], "%0.03f") + " = " + feature_names[order[i]] 
    
    # add a last grouped feature to represent the impact of all the features we didn't show
    if num_features < len(values):
        yticklabels[0] = "%d other features" % (len(values) - num_features + 1)
        remaining_impact = base_values - loc
        if remaining_impact < 0:
            pos_inds.append(0)
            pos_widths.append(-remaining_impact)
            pos_lefts.append(loc + remaining_impact)
            c = dim_g  #colors.red_rgb
        else:
            neg_inds.append(0)
            neg_widths.append(-remaining_impact)
            neg_lefts.append(loc + remaining_impact)
            c = dark_o #colors.blue_rgb

    points = pos_lefts + list(np.array(pos_lefts) + np.array(pos_widths)) + neg_lefts + list(np.array(neg_lefts) + np.array(neg_widths))
    dataw = np.max(points) - np.min(points)
    
    # draw invisible bars just for sizing the axes
    label_padding = np.array([0.1*dataw if w < 1 else 0 for w in pos_widths])
    pl.barh(pos_inds, np.array(pos_widths) + label_padding + 0.02*dataw, left=np.array(pos_lefts) - 0.01*dataw, color=colors.red_rgb, alpha=0)
    label_padding = np.array([-0.1*dataw  if -w < 1 else 0 for w in neg_widths])
    pl.barh(neg_inds, np.array(neg_widths) + label_padding - 0.02*dataw, left=np.array(neg_lefts) + 0.01*dataw, color=colors.blue_rgb, alpha=0)
    
    # define variable we need for plotting the arrows
    head_length = 0.08
    bar_width = 0.8
    xlen = pl.xlim()[1] - pl.xlim()[0]
    fig = pl.gcf()
    ax = pl.gca()
    xticks = ax.get_xticks()
    bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    width, height = bbox.width, bbox.height
    bbox_to_xscale = xlen/width
    hl_scaled = bbox_to_xscale * head_length
    renderer = fig.canvas.get_renderer()
    
    # draw the positive arrows
    for i in range(len(pos_inds)):
        dist = pos_widths[i]
        arrow_obj = pl.arrow(
            pos_lefts[i], pos_inds[i], max(dist-hl_scaled, 0.000001), 0,
            head_length=min(dist, hl_scaled),
            color=dim_g, width=bar_width,
            head_width=bar_width
        )
        
        if pos_low is not None and i < len(pos_low):
            pl.errorbar(
                pos_lefts[i] + pos_widths[i], pos_inds[i], 
                xerr=np.array([[pos_widths[i] - pos_low[i]], [pos_high[i] - pos_widths[i]]]),
                ecolor=dim_g
            )

        txt_obj = pl.text(
            pos_lefts[i] + 0.5*dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                pos_lefts[i] + (5/72)*bbox_to_xscale + dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
                horizontalalignment='left', verticalalignment='center', color=dim_g,
                fontsize=12
            )
    
    # draw the negative arrows
    for i in range(len(neg_inds)):
        dist = neg_widths[i]
        
        arrow_obj = pl.arrow(
            neg_lefts[i], neg_inds[i], -max(-dist-hl_scaled, 0.000001), 0,
            head_length=min(-dist, hl_scaled),
            color=dark_o, width=bar_width,
            head_width=bar_width
        )

        if neg_low is not None and i < len(neg_low):
            pl.errorbar(
                neg_lefts[i] + neg_widths[i], neg_inds[i], 
                xerr=np.array([[neg_widths[i] - neg_low[i]], [neg_high[i] - neg_widths[i]]]),
                ecolor=dark_o
            )
        
        txt_obj = pl.text(
            neg_lefts[i] + 0.5*dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                neg_lefts[i] - (5/72)*bbox_to_xscale + dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
                horizontalalignment='right', verticalalignment='center', color=dark_o,
                fontsize=12
            )

    # draw the y-ticks twice, once in gray and then again with just the feature names in black
    ytick_pos = list(range(num_features)) + list(np.arange(num_features)+1e-8) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    pl.yticks(ytick_pos, yticklabels[:-1] + [l.split('=')[-1] for l in yticklabels[:-1]], fontsize=13)
    
    # put horizontal lines for each feature row
    for i in range(num_features):
        pl.axhline(i, color="#cccccc", lw=0.5, dashes=(1, 5), zorder=-1)
    
    # mark the prior expected value and the model prediction
    pl.axvline(base_values, 0, 1/num_features, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    fx = base_values + values.sum()
    pl.axvline(fx, 0, 1, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    
    # clean up the main axis
    pl.gca().xaxis.set_ticks_position('bottom')
    pl.gca().yaxis.set_ticks_position('none')
    pl.gca().spines['right'].set_visible(False)
    pl.gca().spines['top'].set_visible(False)
    pl.gca().spines['left'].set_visible(False)
    ax.tick_params(labelsize=13)
    #pl.xlabel("\nModel output", fontsize=12)

    # draw the E[f(X)] tick mark
    xmin,xmax = ax.get_xlim()
    ax2=ax.twiny()
    ax2.set_xlim(xmin,xmax)
    ax2.set_xticks([base_values, base_values+1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax2.set_xticklabels(["\n$E[f(X)]$","\n$ = "+format_value(base_values, "%0.03f")+"$"], fontsize=12, ha="left")
    ax2.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)

    # draw the f(x) tick mark
    ax3=ax2.twiny()
    ax3.set_xlim(xmin,xmax)
    ax3.set_xticks([base_values + values.sum(), base_values + values.sum() + 1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax3.set_xticklabels(["$f(x)$","$ = "+format_value(fx, "%0.03f")+"$"], fontsize=12, ha="left")
    tick_labels = ax3.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-10/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(12/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_color("#999999")
    ax3.spines['right'].set_visible(False)
    ax3.spines['top'].set_visible(False)
    ax3.spines['left'].set_visible(False)

    # adjust the position of the E[f(X)] = x.xx label
    tick_labels = ax2.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-20/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(22/72., -1/72., fig.dpi_scale_trans))
    
    tick_labels[1].set_color("#999999")

    # color the y tick labels that have the feature values as gray
    # (these fall behind the black ones with just the feature name)
    tick_labels = ax.yaxis.get_majorticklabels()
    for i in range(num_features):
        tick_labels[i].set_color("#999999")
    
    if show:
        pl.show()

def dbscan_plot(data,eps=0.1,min_samples=50):
    X=data
    X = StandardScaler().fit_transform(X)
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
    core_samples_mask[db.core_sample_indices_] = True
    labels = db.labels_

    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)

    print('Estimated number of clusters: %d' % n_clusters_)
    print('Estimated number of noise points: %d' % n_noise_)
    print("Silhouette Coefficient: %0.3f"
          % metrics.silhouette_score(X, labels))

    # Black removed and is used for noise instead.
    plt.figure(figsize=(10, 10))
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each)
              for each in np.linspace(0, 1, len(unique_labels))]
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black used for noise.
            col = [0, 0, 0, 1]

        class_member_mask = (labels == k)
        
        xy = X[class_member_mask & core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),label = k,
                 markeredgecolor='k', markersize=14)
        
        xy = X[class_member_mask & ~core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                 markeredgecolor='k', markersize=6)
    
    plt.legend(fontsize=15, title_fontsize='40')    
    plt.title('Estimated number of clusters: %d' % n_clusters_)
#    plt.show()
    return labels



def residual(params, x, data):
    alpha = params['alpha']
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H3.3']*alpha+x['H4']*beta+x['H3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H3.3'])+np.std(od['H4'])+np.std(od['H3'])


def residual2(params, x, data):
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H4']*beta+x['H3.3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H4'])+np.std(od['H3.3'])



def twoSampZ(X1, X2):
    from numpy import sqrt, abs, round
    from scipy.stats import norm
    mudiff=np.mean(X1)-np.mean(X2)
    sd1=np.std(X1)
    sd2=np.std(X2)
    n1=len(X1)
    n2=len(X2)
    pooledSE = sqrt(sd1**2/n1 + sd2**2/n2)
    z = ((X1 - X2) - mudiff)/pooledSE
    pval = 2*(1 - norm.cdf(abs(z)))
    return round(pval, 4)

def statistic(dframe):
    return dframe.corr().loc[Var1,Var2]


def draw_umap(data,n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean', title=''
              ,cc=0,rstate=42):
    fit = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric, random_state=rstate, verbose=True
    )
    u = fit.fit_transform(data);
    plt.figure(figsize=(6, 5))
    if n_components == 2:
        plt.scatter(u[:,0], u[:,1], c=cc,s=3,cmap=plt.cm.seismic)
        plt.clim(-5,5)
        plt.colorbar()
    plt.title(title, fontsize=18)
    return u;


def NormMark(data):
    params = Parameters()
    params.add('beta', value=0.1, min=0)
    params.add('gamma', value=0.1, min=0)
    params.add('alpha', value=0.1, min=0)
    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value
    alpha=out.params['alpha'].value
    avMarkers=ddf['H3.3']*alpha+ddf['H4']*beta+ddf['H3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols]=data[EpiCols]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data

def NormMark2(data):
    params = Parameters()
    params.add('beta', value=0.1, min=-1000)
    params.add('gamma', value=0.1, min=-1000)

    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual2, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value

    avMarkers=ddf['H4']*beta+ddf['H3.3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols_M]=data[EpiCols_M]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data






def f(): raise Exception("Found exit()")



def BPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.boxplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.xticks(rotation=90)
        plt.show()   

def VPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.violinplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   


def KPlots(data,NMS,titleSup=''):
    for NN in NMS:
        plt.figure(figsize=(10,10))
        sns.kdeplot(data=data,x=NN,color='blue')
        
#        plt.legend()
        plt.title(""+NN+" "+titleSup)
        plt.show()



def MeanDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

    
def MedDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].median().sort_values(ascending=False)
    dd1=data2[Markers].median().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)    
    
def MeanDistIdU(data1,data2,Markers,title=''):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    
    colors = ['dodgerblue' if x < 0 else 'darkmagenta' for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

def KPlot_Mrk(Mark,titleSup=''):
    plt.figure(figsize=(10,10))
    sns.kdeplot(data=C01,x=Mark,label="C01")
    sns.kdeplot(data=C02,x=Mark,label="C02")
    sns.kdeplot(data=C03,x=Mark,label="C03")
    sns.kdeplot(data=C04,x=Mark,label="C04")
    sns.kdeplot(data=C05,x=Mark,label="C05")
    plt.legend()
    plt.title(""+Mark+" "+titleSup)
    plt.show()
    
    
    
    

def UMAP_Plot(data1,data2,Markers,Set1='C01',Set2='Other',titleSup=''):
    data1=data1.assign(Set=Set1)
    data2=data2.assign(Set=Set2)
    CAll=data1.append(data2).sample(frac=0.1).copy()
    print(CAll)
    X_2d=draw_umap(CAll[Markers],cc=CAll['H3'],min_dist=0.01)
    for NN in NamesAll:
        cc=CAll[NN]#[mask]
        plt.figure(figsize=(6, 5))
        plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                    c=cc, cmap=plt.cm.jet)
    #    cmap = matplotlib.cm.get_cmap('jet')
        plt.colorbar()
    #    plt.clim(-3.5,3.5)
        plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    #    mask=CAllmask[TSNEVar]==True
    #    rgba = cmap(-10)
    #    plt.scatter(X_2d[mask][:,0],X_2d[mask][:,1],s=2,
    #                color=rgba) 
        plt.title(NN+" "+titleSup)
        plt.show()

    plt.figure(figsize=(6, 5))
    mask=CAll.Set==Set1
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='blue', label=Set1)        
    mask=CAll.Set==Set2
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='red', label=Set2)        
    plt.legend()
    plt.show()
       

def DeltaCorr(data1,data2,Markers,titleSup=''):
    params = {'axes.titlesize': 30,
              'legend.fontsize': 20,
              'figure.figsize': (16, 10),
              'axes.labelsize': 20,
              'axes.titlesize': 20,
              'xtick.labelsize': 16,
              'ytick.labelsize': 16,
              'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

    print(titleSup)
    plt.figure(figsize=(20,20))
    matrix=data2[Markers].corr()-data1[Markers].corr()
    g=sns.clustermap(matrix, annot=True, annot_kws={"size":8},
                     cmap=plt.cm.jet,vmin=matrix.min().min(),vmax=matrix.max().max(),linewidths=.1); 
    plt.xticks(rotation=0); 
    plt.yticks(rotation=0); 

    plt.title(titleSup)
    plt.show()
    
    
def DefStyle():
    params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")


def MeanDistReSamp(data1,data2,Markers,title='',clr=['darkgreen','purple'],nsamp=10,f=0.5):
    sns.set_style({'legend.frameon':True})
    diffs=[]
    for i in range(nsamp):  
        D1=data1.sample(frac=f).copy()
        D2=data2.sample(frac=f).copy()
        dd0=D1[Markers].mean()#.sort_values(ascending=False)
        dd1=D2[Markers].mean()#.sort_values()
        diff=(dd1-dd0)#.sort_values(ascending=False)    
        diffs.append(diff)

    Mdiff=np.asarray(diffs)
    D=pd.DataFrame({'M':Mdiff.mean(axis=0),'S':Mdiff.std(axis=0)},index=Markers)    
    
    diffs=D.sort_values(by='M',ascending=False).copy()
    
    
    colors = [clr[0] if x < 0 else clr[1] for x in diffs.M]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs.M, color=colors, alpha=1, linewidth=5)
    plt.errorbar(y=diffs.index,x=diffs.M,xerr=diffs.S,capsize=5,fmt='k.')
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)            
    return(D)

# Load and initialize

In [ ]:
import fcsparser

dir="/Users/ronguy/Dropbox/CyTOF_Breast/CyTOF_Oren_MCF7_Tam/data/"
path=dir+"c16_export_CyTOF2_0_cellcycle_christi_12Jun2023_01_0_Time, 141Pr_EpCAM subset_MCF7_Yael.fcs"

c15=pd.read_csv(dir+"c15_MCF7_NCI_control.csv")
c16=pd.read_csv(dir+"c16_MCF7_NCI_Tam.csv")
c17=pd.read_csv(dir+"c17_MCF7_Yael_control.csv")
c18=pd.read_csv(dir+"c18_MCF7_Yael_Tam.csv")
c19=pd.read_csv(dir+"c19_MCF7_Ori_control.csv")
c20=pd.read_csv(dir+"c20_MCF7_Ori_Tam.csv")

In [ ]:
NamesAll=[
 'K5',
 'H4K20me3',
 'ER',
 'CD49f',
 'CD24',
 'H3K27me3',
 'H3K9me3',
 'H3K9me2',
 'H2Aub',
 'H3.3',
 'H3K64ac',
 'BMI1',
 'ZEB1',
 'H3K27ac',
 'H3K36me3',
 'H3',
 'pH3',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'sma',
 'EpCAM',
 'H3K3me2',
 'H4K16ac',
 'H3K4me1',
 'H4',
 'H3K9ac',
 'CD44',
 'Ki-67',
 'K8-18'
]


EpiCols=[
 'H4K20me3',
 'H3K27me3',
 'H3K9me3',
 'H3K9me2',
 'H2Aub',
 'H3.3',
 'H3K64ac',
 'H3K27ac',
 'H3K36me3',
 'H3',
 'pH3',
 'H3K27me2',
 'H3K4me3',
 'H3K3me2',
 'H4K16ac',
 'H3K4me1',
 'H4',
 'H3K9ac',
]

NamesAll.sort()
EpiCols.sort()

In [ ]:



params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")

In [ ]:
hKWD={'stat':'density','element':'step','fill':False}

In [ ]:
sns.histplot(data=c19,x='H3',color='r',**hKWD)
sns.histplot(data=c19,x='H3.3',color='g',**hKWD)
sns.histplot(data=c19,x='H4',color='b',**hKWD)
plt.plot([5,5],[0,0.01])
plt.xscale('log')

# Gate on H3.3/H4 too low, but also remove outliers 99.99% from all 

In [ ]:
GateColumns=['H3.3','H4','H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data


c15=Gate(c15,"")
c16=Gate(c16,"")
c17=Gate(c17,"")
c18=Gate(c18,"")
c19=Gate(c19,"")
c20=Gate(c20,"")

scFac=5
c15=np.arcsinh(c15/scFac)
c16=np.arcsinh(c16/scFac)
c17=np.arcsinh(c17/scFac)
c18=np.arcsinh(c18/scFac)
c19=np.arcsinh(c19/scFac)
c20=np.arcsinh(c20/scFac)

In [ ]:
len(NamesAll)

In [ ]:
a=[]
DefStyle()
fig, axs = plt.subplots(8, 4, figsize=(15,30))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)
        
for i,N in enumerate(NamesAll):
    print(N)
    sns.histplot(data=c15,x=N,ax=a[i],color='r',label=N,**hKWD)   
    sns.histplot(data=c16,x=N,ax=a[i],color='orange',label=N,**hKWD)    
    sns.histplot(data=c17,x=N,ax=a[i],color='yellow',label=N,**hKWD)    
    sns.histplot(data=c18,x=N,ax=a[i],color='g',label=N,**hKWD)    
    sns.histplot(data=c19,x=N,ax=a[i],color='b',label=N,**hKWD)    
    sns.histplot(data=c20,x=N,ax=a[i],color='magenta',label=N,**hKWD)    
#    a[i].set_title(N)

plt.subplots_adjust(wspace=0.75, hspace=0.7)
#fig.suptitle('MCF7 - Raw Data',y=0.91);
#fig.savefig("Plots/MCF7-Raw.png",dpi=200,bbox_inches='tight')

# Normalize using new method on all intercellular markers

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return d.std()['H3.3']**2+d.std()['H4']**2+d.std()['H3']**2

def NormalizeNew(data):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[EpiCols]=data[EpiCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3.3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.3,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf, ddf,Q,M,M1,M2),method='cg')
    AA=out.params['a'].value

    M=M1*AA+M2*(1-AA)
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[EpiCols]=data[EpiCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:


c15=NormalizeNew2(c15)
c16=NormalizeNew2(c16)
c17=NormalizeNew2(c17)
c18=NormalizeNew2(c18)
c19=NormalizeNew2(c19)
c20=NormalizeNew2(c20)


In [ ]:
from tqdm import tqdm

def DivMean(DB):
    x=DB.copy()
    Mean_Core=x[['H3.3','H4']].mean(axis=1)
    for N in tqdm(EpiCols):
        x[N]=x[N]/Mean_Core
    return x
# Mean_Core=MCF7[['H3.3','H4','H3']].mean(axis=1)
# for N in tqdm(EpiCols):
#     MCF7[N]=MCF7[N]/Mean_Core


c15=DivMean(c15)
c16=DivMean(c16)
c17=DivMean(c17)
c18=DivMean(c18)
c19=DivMean(c19)
c20=DivMean(c20)


In [ ]:
Control=pd.concat([c15,c17,c19]).copy()
Tam=pd.concat([c16,c18,c20]).copy()

In [ ]:
aaaa=pd.concat([
    c15.sample(10000,replace=True),
#    c16.sample(10000,replace=True),
    c17.sample(10000,replace=True),
#    c18.sample(10000,replace=True),
    c19.sample(10000,replace=True),
#    c20.sample(10000,replace=True),
]).copy()
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)


c15=(c15-m)/s
c16=(c16-m)/s
c17=(c17-m)/s
c18=(c18-m)/s
c19=(c19-m)/s
c20=(c20-m)/s


aaaa=pd.concat([
    Control.sample(10000,replace=True),
    Tam.sample(10000,replace=True),
]).copy()
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)


Control=(Control-m)/s
Tam=(Tam-m)/s


params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")

In [ ]:
EPC=EpiCols.copy()
EPC.remove('H3.3')
EPC.remove('H3')
EPC.remove('H4')


In [ ]:
CAll=pd.concat([
    c15.sample(4000,replace=False),c17.sample(4000,replace=False),c19.sample(4000,replace=False),
]).copy()




In [ ]:
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()


In [ ]:
labels=dbscan_plot(X_2d,eps=0.1,min_samples=50)

In [ ]:
import xgboost as xgb
XG=xgb.XGBClassifier()
lab=labels+1
XG.fit(CAll[EPC],lab)

In [ ]:
CAll=pd.concat([
    c15.sample(4000,replace=False),c17.sample(4000,replace=False),c19.sample(4000,replace=False),
]).copy()
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)



In [ ]:
p=XG.predict(CAll[EPC])

In [ ]:
for i in range(6):
    m=p==i
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1)

In [ ]:
Control.shape

In [ ]:
AllC=pd.concat([c15,c17,c19]).copy()
pred=XG.predict(AllC[EPC])

In [ ]:
Control['pred']=pred

In [ ]:
XG.fit(AllC[EPC],pred)

In [ ]:
q=XG.predict(Tam[EPC])

In [ ]:
Tam['pred']=q

In [ ]:
Control['Line']='Control'
Tam['Line']='Tam'

In [ ]:
Colors={
    'NCI Control':'lightgreen',
    'NCI Tam':'green',
    'Yael Control':'pink',
    'Yael Tam':'magenta',
    'Ori Control':'cyan',
    'Ori Tam':'blue',
    'Control':'green',
    'Tam':'magenta'
}

In [ ]:
m=Control.pred!=0
Control=Control[m]
m=Tam.pred!=0
Tam=Tam[m]

In [ ]:
CAll=pd.concat([
    Control.sample(10000,replace=False),
    Tam.sample(10000,replace=False),
]).copy()


In [ ]:
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.01,n_neighbors=60,rstate=42)


In [ ]:
for i in CAll.Line.unique():
    plt.figure()
    m=CAll.Line==i
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,label=i)

plt.legend()

In [ ]:
for NN in NamesAll:#['Gata3','Cytokeratin','H3S28p']:
    Var=NN
    TSNEVar=NN
    
    vmn=CAll[NN].quantile(0.01)
    vmx=CAll[NN].quantile(0.99)

    fig,ax=plt.subplots(ncols=2,nrows=1,figsize=(20, 5))
    m=CAll.Line=='Control'
    cc=CAll[m][NN]#[mask]
    ax[0].scatter(X_2d[m,0],X_2d[m,1],s=2,
                c=cc, cmap=plt.cm.seismic,vmin=vmn, vmax=vmx)
#    ax[0].colorbar()
#    ax[0].set_clim(cc.quantile(0.01),cc.quantile(0.99))
    m=CAll.Line=='Tam'
    cc=CAll[m][NN]#[mask]
    ax[1].scatter(X_2d[m,0],X_2d[m,1],s=2,
                c=cc, cmap=plt.cm.seismic,vmin=vmn, vmax=vmx)
#    ax[1].colorbar()
    

#    ax[1].clim(cc.quantile(0.01),cc.quantile(0.99))

    ax[0].set_title("Control "+TSNEVar)
    ax[1].set_title("Tam "+TSNEVar)

#    plt.xlabel('UMAP 0');
#    plt.ylabel('UMAP 1');
#    plt.savefig('Plots/MCF7_'+NN+'.png',dpi=200,bbox_inches='tight')
    fig.savefig('Plots/MCF7_Control_Tam_'+NN+'.png',dpi=200,bbox_inches='tight')
    plt.show()

In [ ]:
vmx

In [ ]:
from  sklearn.svm import SVC

In [ ]:
m=CAll.Line=='Control'
labels=dbscan_plot(X_2d[m],eps=0.1,min_samples=60)

In [ ]:
SV=SVC()
SV.fit(X_2d[m],labels)

In [ ]:
lab=SV.predict(X_2d)

In [ ]:
for i in range(4):
    m2=lab==i
    plt.scatter(X_2d[m2,0],X_2d[m2,1],s=1,label=i)
plt.legend(markerscale=10)

In [ ]:
CAll['Clust']=lab

In [ ]:
m=CAll.Line=='Control'
print('Control')
print((CAll[m].groupby('Clust').count()['H3'])/m.sum())
m=CAll.Line=='Tam'
print('Tam')
print((CAll[m].groupby('Clust').count()['H3'])/m.sum())


In [ ]:
CAll['U0']=X_2d[:,0]
CAll['U1']=X_2d[:,1]

In [ ]:
m=CAll.Clust.isin([0,1]) 

In [ ]:
(CAll[m].groupby(['Line','Clust']).mean())[['U0','U1']]

In [ ]:
Points=(CAll[m].groupby(['Line','Clust']).mean())[['U0','U1']].values

In [ ]:
for i in range(2):
    m2=lab==i
    plt.scatter(X_2d[m2,0],X_2d[m2,1],s=1,alpha=0.3)
for i in range(2):
    plt.scatter(Points[i,0],Points[i,1],s=100,color='green',edgecolors='k')
    plt.scatter(Points[i+2,0],Points[i+2,1],s=100,color='magenta',edgecolors='k')
    

In [ ]:
m=CAll.Clust.isin([0,1])
Points=(CAll[m].groupby(['Line']).mean())[['U0','U1']].values

In [ ]:
Points

In [ ]:
for i in range(2):
    m2=lab==i
    plt.scatter(X_2d[m2,0],X_2d[m2,1],s=1,alpha=0.3)
for i in range(1):
    plt.scatter(Points[i,0],Points[i,1],s=100,color='green',edgecolors='k')
    plt.scatter(Points[i+1,0],Points[i+1,1],s=100,color='magenta',edgecolors='k')
    

In [ ]:
for NN in EPC1:#['Gata3','Cytokeratin','H3S28p']:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title(""+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');
#    plt.savefig('Plots/MCF7_'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
m=CAll.Line=='Control'
print('Control')
print((CAll[m].groupby('pred').count()['H3'])/m.sum())
m=CAll.Line=='Tam'
print('Tam')
print((CAll[m].groupby('pred').count()['H3'])/m.sum())


In [ ]:
for i in range(1,5):
    m=CAll.pred==i
    print('Cluster ',i)
    print((CAll[m].groupby('Line').count()['H3'])/m.sum())
    

In [ ]:

X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

# plotting the clusters using the first 2 dimentions of the data
for c in np.unique(cluster_index):
    plt.scatter(X_2d[cluster_index == c, 0],
                X_2d[cluster_index == c, 1], label='cluster='+str(c), alpha=.7)


plt.legend();


In [ ]:
labels=dbscan_plot(X_2d,eps=0.2,min_samples=60)

In [ ]:
m=labels!=0

In [ ]:
CAll=CAll[m]

In [ ]:
EPC1=['H3K9me2','H3K9me3','H3K9ac','H3K4me3','H3K36me3','H4K20me3']

In [ ]:
X_2d=draw_umap(CAll[EPC1],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
CAll['Clust']=labels

In [ ]:
m=CAll.Clust>-1
Mat=CAll[m].groupby('Clust').mean(numeric_only=True)

In [ ]:
for L in CAll.Clust.unique():
    m=CAll.Clust==L
#    plt.figure()
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,label="Cluster "+str(L))

plt.legend(bbox_to_anchor=(1,1),markerscale=10)
plt.savefig('Plots/MCF7_Clust.png',dpi=200,bbox_inches='tight')

In [ ]:
plt.figure(figsize=(5,10))
sns.clustermap(Mat.T,cmap=plt.cm.seismic,annot=True,yticklabels=True,vmin=-2,vmax=2)
plt.savefig('Plots/MCF7_HeatMap.png',dpi=200,bbox_inches='tight')

In [ ]:
for L in CAll.Line.unique():
    m=CAll.Line==L
#    plt.figure()
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)

plt.legend(bbox_to_anchor=(1,1),markerscale=10)

In [ ]:
for NN in NamesAll:#['Gata3','Cytokeratin','H3S28p']:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title(""+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');
#    plt.savefig('Plots/MCF7_'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
CAll=MCF7.sample(10000).copy()
X_2d=draw_umap(CAll[EPC],cc=CAll['Gata3'],min_dist=0.1,n_neighbors=200,rstate=42)
plt.show()

In [ ]:
CAll=MCF7.sample(10000).copy()
X_2d = trimap.TRIMAP(verbose=True,n_iters=1000,n_inliers=12,n_outliers=10).fit_transform(CAll[EPC].values)
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c=CAll['Gata3'])

In [ ]:
labels=dbscan_plot(X_2d,eps=0.19,min_samples=60)

In [ ]:
labels[labels==5]=4

In [ ]:
CAll['Cl']=labels

In [ ]:
#CAll=CAll[CAll.Cl>-1]

In [ ]:
Mat=CAll[CAll.Cl>-1].groupby('Cl').mean(numeric_only=True)

In [ ]:
plt.figure(figsize=(5,30))
sns.clustermap(np.round(Mat.T,2),cmap=plt.cm.seismic,annot=True,vmin=-2,vmax=2,row_cluster=True,yticklabels=True,figsize=(10,15))
plt.savefig('Plots/MCF7_Clust.png',dpi=200,bbox_inches='tight')

In [ ]:
for C in range(6):
    m=CAll.Cl==C
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,label=C)    

plt.legend(markerscale=10)
plt.savefig('Plots/MCF7_Clust_ID.png',dpi=200,bbox_inches='tight')


In [ ]:
(CAll.groupby('Phase').count()/len(CAll))['H3']

In [ ]:
for C in range(5):
    m=CAll.Cl==C
    print('Cluster '+str(C))
    print(CAll[m].groupby('Phase').count()['H3']/m.sum())

In [ ]:
m=CAll.Cl>-1
(CAll[m].groupby('Cl').count())/m.sum()

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("(All Marks) "+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');
    plt.savefig('Plots/MCF7_'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

import scanpy as sc
import anndata
CN=anndata.AnnData(CAll[NamesAll],dtype=np.float32)
CN.obsm['X_umap']=X_2d

from matplotlib.pyplot import rc_context
sc.set_figure_params(scanpy=True, fontsize=13)
with rc_context({'figure.figsize': (3,3),
                'font.size': 8.0}):
    sc.pl.umap(CN,color=NamesAll,cmap=plt.cm.seismic,add_outline=True,ncols=5,show=False,alpha=.3,legend_loc=None
              ,)

#plt.savefig('Plots/MCF7_All.png',dpi=200,bbox_inches='tight')


In [ ]:
EPC=EpiCols.copy()
EPC.remove('H3.3')
EPC.remove('H3')
EPC.remove('H4')

In [ ]:
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
import scanpy as sc
import anndata
CN=anndata.AnnData(CAll[NamesAll],dtype=np.float32)
CN.obsm['X_umap']=X_2d

from matplotlib.pyplot import rc_context
sc.set_figure_params(scanpy=True, fontsize=13)
with rc_context({'figure.figsize': (3,3),
                'font.size': 8.0}):
    sc.pl.umap(CN,color=NamesAll,cmap=plt.cm.seismic,add_outline=True,ncols=5,show=False,alpha=.3,legend_loc=None
              ,)

#plt.savefig('Plots/MCF7_Epi.png',dpi=200,bbox_inches='tight')


In [ ]:
EPC1=['H3K27me2',
'H3K4me3',
'H3K36me2',
'H3K9me2',
'H4K16ac',
'H3K64ac',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'yH2A.X',
]
EPC2=['H3K9me2', 'H3K4me3',
      'H3K36me3', 'H3K9me3', 'H3K9ac']

In [ ]:
X_2d=draw_umap(CAll[EPC1],cc=CAll['H4'],min_dist=0.001,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
CN=anndata.AnnData(CAll[NamesAll],dtype=np.float32)
CN.obsm['X_umap']=X_2d

from matplotlib.pyplot import rc_context
sc.set_figure_params(scanpy=True, fontsize=13)
with rc_context({'figure.figsize': (3,3),
                'font.size': 8.0}):
    sc.pl.umap(CN,color=NamesAll,cmap=plt.cm.seismic,add_outline=True,ncols=5,show=False,alpha=.3,legend_loc=None
              ,)

plt.savefig('Plots/MCF7_Epi_1.png',dpi=200,bbox_inches='tight')


In [ ]:
X_2d=draw_umap(CAll[EPC2],cc=CAll['H4'],min_dist=0.001,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
CN=anndata.AnnData(CAll[NamesAll],dtype=np.float32)
CN.obsm['X_umap']=X_2d

from matplotlib.pyplot import rc_context
sc.set_figure_params(scanpy=True, fontsize=13)
with rc_context({'figure.figsize': (3,3),
                'font.size': 8.0}):
    sc.pl.umap(CN,color=NamesAll,cmap=plt.cm.seismic,add_outline=True,ncols=5,show=False,alpha=.3,legend_loc=None
              ,)

plt.savefig('Plots/MCF7_Epi_2.png',dpi=200,bbox_inches='tight')
